<h1>Model Training and Evaluation</h1>
<hr/>
<p>This notebook trains the final regression model and evaluates it against a baseline model. The workflow follows the original notebook style by comparing multiple evaluation metrics and explaining what each result means.</p>


<hr/>
<h1>1.&nbsp;&nbsp;&nbsp;&nbsp;Importing Required Modules</h1>
<hr/>
<p>The training code is imported from the project package. This ensures that the notebook, command-line interface, tests, and CI workflow all use the same modeling logic.</p>


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
import json
import pandas as pd

from us_housing_price_prediction.modeling import save_metrics, save_model, train_and_evaluate

pd.set_option("display.float_format", "{:.4f}".format)


<hr/>
<h1>2.&nbsp;&nbsp;&nbsp;&nbsp;Training the Final Model</h1>
<hr/>
<p>The final model is trained using a scikit-learn pipeline. The pipeline performs feature engineering, preprocessing, encoding, and regression modeling in one reproducible workflow. This is better than manually transforming the full dataset before splitting because it reduces the risk of data leakage.</p>


In [ ]:
model, result = train_and_evaluate()
model_path = save_model(model)
metrics_path = save_metrics(result)

print(f"Model saved to: {model_path}")
print(f"Metrics saved to: {metrics_path}")


<p>With reference to the output above, the final model and evaluation metrics have been saved successfully. Saving the model allows the prediction workflow to reuse the trained pipeline without retraining every time.</p>


<hr/>
<h1>3.&nbsp;&nbsp;&nbsp;&nbsp;Baseline and Final Model Comparison</h1>
<hr/>
<p>A baseline model is required so that the final model can be evaluated meaningfully. In this project, the baseline predicts the median price. The final voting regressor should outperform this baseline across key regression metrics.</p>


In [ ]:
comparison = pd.DataFrame(
    [result.metrics, result.baseline_metrics],
    index=["VotingRegressor", "MedianBaseline"],
)
comparison


<p>With reference to the comparison table, the final model should have a higher R2 score and lower error metrics such as MAE, MSE, RMSE, and MAPE compared to the baseline. This indicates that the model has learned useful patterns beyond simply predicting the median price.</p>


<hr/>
<h1>4.&nbsp;&nbsp;&nbsp;&nbsp;Cross-Validation Evaluation</h1>
<hr/>
<p>Cross-validation is used to estimate how stable the model performance is across different train/test splits. This is especially important because the dataset is small, so a single split may not fully represent model performance.</p>


In [ ]:
pd.DataFrame([result.cross_validation])


<p>The cross-validation summary provides the mean and standard deviation of performance. A stable model should have reasonable average performance and not vary excessively across folds.</p>


<hr/>
<h1>5.&nbsp;&nbsp;&nbsp;&nbsp;Paired T-Test Against Baseline</h1>
<hr/>
<p>In addition to normal regression metrics, a paired t-test is used to compare the absolute errors from the final model and the baseline on the same test rows. This supports the evaluation by checking whether the model's improvement over the baseline is statistically meaningful.</p>


In [ ]:
pd.DataFrame([result.residual_test])


<p>If the p-value is below 0.05 and the mean error delta is negative, it suggests that the final model has significantly lower absolute error than the baseline on the test set. This strengthens the justification for using the final model instead of the baseline.</p>


<hr/>
<h1>6.&nbsp;&nbsp;&nbsp;&nbsp;Feature Significance Summary</h1>
<hr/>
<p>The feature significance table summarizes the p-value tests conducted on the raw features. This is useful for interpretation, but it should be considered supporting evidence rather than the sole basis for model selection.</p>


In [ ]:
pd.DataFrame(result.feature_significance)


<hr/>
<h1>7.&nbsp;&nbsp;&nbsp;&nbsp;Full Metrics JSON</h1>
<hr/>
<p>The final cell prints the full metrics artifact in JSON format. This is the same type of output saved to <code>reports/metrics.json</code>, making the results easy to track and compare across runs.</p>


In [ ]:
print(json.dumps(result.to_dict(), indent=2))
